[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# A Complete API


## What you will be able to do

Give a FastAPI app every method on a resource: `POST` answered with `201` and a `Location`, `GET` for
one item and for all of them, `PUT` to replace, `PATCH` to change part, and `DELETE` with `204`. Find
an item once for every route with a dependency, read a request header, answer every error in one
shape, and check a rebuilt API against the one it copies, request by request.


## The idea

### The problem

Every answer the practice API gave in this guide was a choice. A created plan got `201` and its
address in `Location`, a removed one got `204` and no body, a second removal got `404`, a page of
readings came with a `Link` to the next, and anything that went wrong came back as `{"error": ...}`.
A client depends on those choices: the client in the **A Real Client** notebook follows `Link`, and
turns an error's body into an exception. A server that chose differently would break such a client
without failing a single request of its own.

An API of your own has to make those choices on purpose, for every method on every resource. And a
copy of an API has to be checked against the original, request by request, because nothing in
either server shows where the two disagree.

### What a complete API is

> A **complete API** gives each resource every method it needs, with a deliberate status code for
> every outcome. `POST` to a collection creates an item and answers `201 Created`, with the item's
> address in `Location`. `GET` reads one item or all of them. `PUT` replaces an item and `PATCH`
> changes part of it, and both answer `200` with the result. `DELETE` removes an item and answers
> `204 No Content`, with no body. In FastAPI, `status_code=` in a route's decorator sets its status
> for success, a parameter typed `Response` lets the route set headers, and a **dependency**, a
> function named in `Depends`, runs before the route to do work that several routes share, such as
> finding an item or answering `404`. One exception handler can answer every error, raised by a route
> or by FastAPI itself, in one shape.

### Why it works that way

- **A status code is part of the answer.** A client reads `201` as created, `204` as done with
  nothing to send, and `404` as not there, before it reads any body, as the **Status Codes**
  notebook did.
- **`Location` saves a client a guess.** The server chose the new plan's id, so the server says where
  the plan is, and a client follows the header instead of building the address.
- **A dependency is written once, and runs for every route that names it.** Four routes act on one
  plan, and all four need the plan or a `404`. `Depends(plan_or_404)` does that before any of them
  runs, so none of them repeats it, and none can forget it.
- **`PATCH` changes only what was sent.** A model whose fields all have defaults cannot tell a field
  that was sent from one that was not, so `model_dump(exclude_unset=True)` keeps only the fields the
  request set.
- **One handler, one shape.** Routes raise FastAPI's `HTTPException`, and Starlette, the framework
  FastAPI is built on, raises its own for a path that matches no route. A handler for Starlette's
  class answers both.
- **A copy is checked against its original.** The same requests, sent to both servers with their
  status codes, bodies and headers compared, find every place where the copy differs, including the
  places nobody thought to test.

### Where you will meet this

GitHub's REST API creates an issue with `POST /repos/{owner}/{repo}/issues` and answers `201`, and
deletes an issue comment with `DELETE` and answers `204`. Stripe's API updates a customer with a
`POST` to the customer's address, and leaves any parameter it was not sent unchanged, as a `PATCH`
would. FastAPI's documentation notes that many teams use only `PUT`, even for partial updates. The
methods are conventions that an API documents, and its clients read that documentation. The practice
API makes the choices this notebook copies, and the **Hosting an API** notebook puts an app like this
one on a public address.

### What this notebook covers

- `status_code=` and `fastapi.status`, and a `Location` header set through a `Response` parameter
- `GET` for one plan and for every plan, with the plan found by a dependency
- `PUT` to replace a plan, and `PATCH` to change part of one with `exclude_unset=True`
- `DELETE`, answered with `204` and no body
- A request header read with `Header`, for an `Idempotency-Key`
- Every error in one shape, from a handler for Starlette's `HTTPException`
- Pages of readings, with a `Link` header built from the request
- The practice API rebuilt, and compared with the original request by request
- Five errors, from a dependency called instead of named to a created plan answered with `200`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import itertools

import requests
from fastapi import FastAPI, Response, status
from pydantic import BaseModel

import practice_api


class Plan(BaseModel):
    name: str
    latitude: float
    longitude: float


app = FastAPI()
plan_ids = itertools.count(1)


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response):
    created = {"id": next(plan_ids), **plan.model_dump()}
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
body = {"name": "Alta", "latitude": 69.97, "longitude": 23.27}
for url in [f"{practice_api.start()}/network/plans", f"{app_url}/network/plans"]:
    response = requests.post(url, json=body, timeout=10)
    print(response.status_code, response.headers["Location"], response.json())
```

```
201 /network/plans/1 {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27}
201 /network/plans/1 {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27}
```

The practice API's answer first, then the app's: the same status code, the same `Location`, and the
same body, from a route that chose all three.


## Setup

Twenty-six imports, the last of them the practice API.

- `requests` sends every request, to the apps and to the practice API
- `FastAPI` makes an app
- `status` names the status codes, as `status.HTTP_201_CREATED`
- `Response` is the parameter type through which a route sets headers
- `Request` is the parameter type that gives a route the request itself
- `Depends` names a dependency for a route
- `Header` reads a request header into a parameter
- `Query` puts rules on a query parameter
- `HTTPException` is how a route or a dependency answers with an error
- `StarletteHTTPException` is the class that every `HTTPException` extends, which a handler catches
- `RequestValidationError` is what FastAPI raises for a request that breaks the rules
- `JSONResponse` is the response an exception handler sends
- `BaseModel` declares the models for a plan and a change to one
- `Field` puts rules on their fields
- `ConfigDict` forbids fields a model does not declare
- `Annotated` attaches `Depends`, `Header` and `Query` to a parameter's type
- `date` is the type of a plan's `opens`
- `itertools` counts plan ids, with `count`
- `math` rounds the number of pages up, with `ceil`
- `urlencode` writes the query of every address in a `Link` header
- `uuid` makes an idempotency key
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell whether the notebook is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, and its `serve` runs the apps in this notebook

The practice API has no plans whenever this cell runs, and its plan ids start at 1. The last section
compares its answers with an app's, so run the notebook from the top.


In [1]:
import importlib
import itertools
import math
import sys
import urllib.request
import uuid
from datetime import date
from pathlib import Path
from typing import Annotated
from urllib.parse import urlencode

import requests
from fastapi import Depends, FastAPI, Header, HTTPException, Query, Request, Response, status
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field
from starlette.exceptions import HTTPException as StarletteHTTPException

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### A status code for every outcome, and a Location header

A route's status code for success belongs to its decorator, as `status_code=`, and `fastapi.status`
names every code, so `status.HTTP_201_CREATED` says what it means, and an editor can complete it. A
header is set through a parameter typed `Response`: FastAPI passes the route a temporary response,
and copies the headers the route sets on it into the response it sends. A created plan gets `201` and
its address in `Location`, as the practice API sends them. `Plan` is the strict model from the
**Validating Requests** notebook, `plan_from` puts a plan's fields in the practice API's order and
leaves out a field that is `None`, and `plan_ids` counts ids from 1, never giving one out twice:


In [2]:
class Plan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=1, max_length=50)
    latitude: float = Field(ge=-90, le=90, strict=True)
    longitude: float = Field(ge=-180, le=180, strict=True)
    elevation_m: float | None = Field(default=None, strict=True)
    opens: date | None = None


FIELDS = ("name", "latitude", "longitude", "elevation_m", "opens")


def plan_from(plan_id, fields):
    """A plan: its id, then its fields in their usual order, leaving out any field that is None."""
    return {"id": plan_id, **{field: fields[field] for field in FIELDS if fields.get(field) is not None}}


plans = {}
plan_ids = itertools.count(1)
app = FastAPI()


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response):
    created = plan_from(next(plan_ids), plan.model_dump(mode="json"))
    plans[created["id"]] = created
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
body = {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "elevation_m": 8}
response = requests.post(f"{app_url}/network/plans", json=body, timeout=10)

print(response.status_code, "| Location:", response.headers["Location"], "|", response.json())
print(status.HTTP_201_CREATED, status.HTTP_204_NO_CONTENT, status.HTTP_404_NOT_FOUND)


201 | Location: /network/plans/1 | {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': 8.0}
201 204 404


The status came from the decorator and the header from the temporary response, and the names in
`status` are plain numbers, as the last line shows. `model_dump(mode="json")` turns every field into
a value JSON can hold, such as a date into text, so what `plans` keeps is what the response sends.
The elevation came back as `8.0`, because a strict `float` field still takes a whole number, and
keeps it as a float.

### One plan and every plan: GET, and a dependency

`GET /network/plans` lists the plans in order of id. `GET /network/plans/{plan_id}` sends one, and
`PUT`, `PATCH` and `DELETE` will act on one, so all four need the plan or a `404`. A **dependency**
does that once. `plan_or_404` takes `plan_id` from the path, as a route would, and raises the `404`
or returns the plan, and `Depends(plan_or_404)` in a route's parameter makes FastAPI call it before
the route runs, and pass the route what it returned:


In [3]:
def plan_or_404(plan_id: str):
    """The plan with this id, or a 404 for a plan that does not exist."""
    plan = plans.get(int(plan_id)) if plan_id.isdigit() else None
    if plan is None:
        raise HTTPException(status_code=404, detail=f"no plan has the id {plan_id}")
    return plan


app = FastAPI()


@app.get("/network/plans")
def list_plans():
    return [plans[plan_id] for plan_id in sorted(plans)]


@app.get("/network/plans/{plan_id}")
def read_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    return plan


app_url = practice_api.serve(app)
for path in ["/network/plans", "/network/plans/1", "/network/plans/7", "/network/plans/abc"]:
    response = requests.get(f"{app_url}{path}", timeout=10)
    print(response.status_code, path, "|", response.json())


200 /network/plans | [{'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': 8.0}]
200 /network/plans/1 | {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27, 'elevation_m': 8.0}
404 /network/plans/7 | {'detail': 'no plan has the id 7'}
404 /network/plans/abc | {'detail': 'no plan has the id abc'}


`read_plan` never saw a request for a plan that does not exist: the dependency answered `404` before
the route ran. `plan_id` is text, not an `int`, so `abc` gets the same `404` as `7`, as it does from
the practice API, where an `int` would have answered `422`. FastAPI's documentation lists shared
logic among the uses of a dependency, along with sharing database connections and enforcing
security.

### Replacing a plan: PUT

`PUT` replaces all of a plan with its body, so the route takes a whole `Plan`, and the plan that the
dependency found:


In [4]:
app = FastAPI()


@app.put("/network/plans/{plan_id}")
def replace_plan(new: Plan, plan: Annotated[dict, Depends(plan_or_404)]):
    plans[plan["id"]] = plan_from(plan["id"], new.model_dump(mode="json"))
    return plans[plan["id"]]


app_url = practice_api.serve(app)
whole = {"name": "Alta", "latitude": 69.96, "longitude": 23.24}
for path in ["/network/plans/1", "/network/plans/7"]:
    response = requests.put(f"{app_url}{path}", json=whole, timeout=10)
    print(response.status_code, path, "|", response.json())


200 /network/plans/1 | {'id': 1, 'name': 'Alta', 'latitude': 69.96, 'longitude': 23.24}
404 /network/plans/7 | {'detail': 'no plan has the id 7'}


The elevation was not in the body, so the plan no longer has one: a `PUT` body is the whole plan, as
the **Sending Data** notebook found from the client's side. The route answered `200`, FastAPI's
default, with the plan as it now is, and a plan that does not exist got the dependency's `404`.

### Changing part of a plan: PATCH

A `PATCH` body names only the fields it changes, so its model needs every field to be optional, and
a way to tell a field that was sent from one that was not. `PlanChange` gives every field a default
of `None`, and `model_dump(exclude_unset=True)` keeps only the fields the request set, which is how
FastAPI's documentation makes a partial update. A default is not checked against its field's type,
so `name`, `latitude` and `longitude` may be left out but not sent as `null`, while `elevation_m` and
`opens` take `null`, which removes them, as the practice API does:


In [5]:
class PlanChange(BaseModel):
    """A change to a plan: any of its fields, none of them required."""
    model_config = ConfigDict(extra="forbid")

    name: str = Field(default=None, min_length=1, max_length=50)
    latitude: float = Field(default=None, ge=-90, le=90, strict=True)
    longitude: float = Field(default=None, ge=-180, le=180, strict=True)
    elevation_m: float | None = Field(default=None, strict=True)
    opens: date | None = None


app = FastAPI()


@app.patch("/network/plans/{plan_id}")
def change_plan(change: PlanChange, plan: Annotated[dict, Depends(plan_or_404)]):
    sent = change.model_dump(mode="json", exclude_unset=True)
    plans[plan["id"]] = plan_from(plan["id"], {**plan, **sent})
    return plans[plan["id"]]


app_url = practice_api.serve(app)
for change in [{"elevation_m": 20}, {"elevation_m": None, "opens": "2027-06-01"}, {"latitude": None}]:
    response = requests.patch(f"{app_url}/network/plans/1", json=change, timeout=10)
    print(response.status_code, change, "->", response.json())


200 {'elevation_m': 20} -> {'id': 1, 'name': 'Alta', 'latitude': 69.96, 'longitude': 23.24, 'elevation_m': 20.0}
200 {'elevation_m': None, 'opens': '2027-06-01'} -> {'id': 1, 'name': 'Alta', 'latitude': 69.96, 'longitude': 23.24, 'opens': '2027-06-01'}
422 {'latitude': None} -> {'detail': [{'type': 'float_type', 'loc': ['body', 'latitude'], 'msg': 'Input should be a valid number', 'input': None}]}


The first change set the elevation and kept everything else. The second removed the elevation with
`null` and added `opens`, and `plan_from` built the result in the usual order. The third sent
`latitude` as `null`, which a plan cannot be without, and got `422`, as it would from the practice
API, which says `is required`.

### Removing a plan: DELETE, and 204

A removed plan has nothing left to send, so the route answers `204 No Content`, which FastAPI's
documentation says must not have a body. The route returns nothing, and FastAPI sends no body:


In [6]:
app = FastAPI()


@app.delete("/network/plans/{plan_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    del plans[plan["id"]]


app_url = practice_api.serve(app)
for attempt in ["first", "second"]:
    response = requests.delete(f"{app_url}/network/plans/1", timeout=10)
    print(attempt, response.status_code, "| body:", response.content)
print("plans left:", plans)


first 204 | body: b''
second 404 | body: b'{"detail":"no plan has the id 1"}'
plans left: {}


The second `DELETE` found no plan, and the dependency answered `404`, as the practice API answered
the second `DELETE` in the **Sending Data** notebook: the plan is gone either way.

### A header the route reads: Idempotency-Key

`Header()` makes a parameter a request header. `idempotency_key` reads `Idempotency-Key`, because
FastAPI turns the underscores in a parameter's name into hyphens, and header names ignore case.
`saved_posts` keeps, for each key, the fields it came with and the plan it created, so a repeat gets
the same plan back with `Idempotent-Replayed: true`, and the same key with another body gets `422`,
as the practice API answers them:


In [7]:
saved_posts = {}          # an Idempotency-Key -> the fields it came with, and the plan it created
app = FastAPI()


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response, idempotency_key: Annotated[str | None, Header()] = None):
    fields = plan.model_dump(mode="json")
    if idempotency_key in saved_posts:
        sent, created = saved_posts[idempotency_key]
        if sent != fields:
            raise HTTPException(status_code=422, detail="this Idempotency-Key was already used with a different body")
        response.headers["Idempotent-Replayed"] = "true"
    else:
        created = plan_from(next(plan_ids), fields)
        plans[created["id"]] = created
        if idempotency_key is not None:
            saved_posts[idempotency_key] = (fields, created)
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
key = {"Idempotency-Key": str(uuid.uuid4())}
karasjok = {"name": "Karasjok", "latitude": 69.47, "longitude": 25.51}
for body in [karasjok, karasjok, {**karasjok, "elevation_m": 129}]:
    response = requests.post(f"{app_url}/network/plans", json=body, headers=key, timeout=10)
    print(response.status_code, "| replayed:", response.headers.get("Idempotent-Replayed"), "|", response.json())
print("plans:", len(plans))


201 | replayed: None | {'id': 2, 'name': 'Karasjok', 'latitude': 69.47, 'longitude': 25.51}
201 | replayed: true | {'id': 2, 'name': 'Karasjok', 'latitude': 69.47, 'longitude': 25.51}
422 | replayed: None | {'detail': 'this Idempotency-Key was already used with a different body'}
plans: 1


The second request created nothing, and got plan 2 again, marked as replayed. Its id is 2 rather than
1, because `plan_ids` never gives an id out twice, and the plan that had 1 was removed. The third
request sent the same key with an elevation added, and was refused.

### Every error in one shape

The practice API answers every error as `{"error": ...}`. Starlette, which FastAPI is built on, has
an `HTTPException` of its own, which FastAPI's `HTTPException` extends and which FastAPI raises
itself for a path that matches no route or a method that a path does not take. FastAPI's
documentation advises registering a handler for Starlette's class, so that it answers both. This
handler passes the error's headers on, so a `405` keeps its `Allow`:


In [8]:
app = FastAPI()


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    """Answer every HTTPException, a route's or FastAPI's own, as the practice API does."""
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/network/plans/{plan_id}")
def read_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    return plan


app_url = practice_api.serve(app)
for method, path in [("GET", "/network/plans/7"), ("GET", "/network/nothing"), ("DELETE", "/network/plans/2")]:
    response = requests.request(method, f"{app_url}{path}", timeout=10)
    print(method, path, "|", response.status_code, response.json(), "| Allow:", response.headers.get("Allow"))


GET /network/plans/7 | 404 {'error': 'no plan has the id 7'} | Allow: None
GET /network/nothing | 404 {'error': 'Not Found'} | Allow: None
DELETE /network/plans/2 | 405 {'error': 'Method Not Allowed'} | Allow: GET


All three came back as `{"error": ...}`: the dependency's `404`, FastAPI's `404` for a path with no
route, and its `405` for a method this app's `/network/plans/{plan_id}` does not take. The last
section's app also answers a request that breaks a model in the practice API's shape, with the
handler from the **Validating Requests** notebook.

### Pages with a Link header

The practice API sends its readings a page at a time, with `page`, `per_page` and `total` in the body
and a `Link` header to the other pages, as the **Pagination** notebook read them. A parameter typed
`Request` gives a route the request itself: its `base_url`, for the addresses in `Link`, and its
`query_params`, so that every link keeps the query the request came with:


In [9]:
app = FastAPI()


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/network/readings")
def readings(request: Request, response: Response, station: str | None = None,
             page: Annotated[int, Query(ge=1)] = 1, per_page: Annotated[int, Query(ge=1)] = 30):
    """Readings a page at a time, with a Link header to the other pages."""
    if station is not None and station not in practice_api.STATIONS:
        raise HTTPException(status_code=400, detail=f"no station has the id {station!r}")
    per_page = min(per_page, 100)                                    # more than 100 gets 100
    chosen = [reading for reading in practice_api.READINGS if station in (None, reading["station"])]
    last = max(1, math.ceil(len(chosen) / per_page))
    address, query = str(request.base_url).rstrip("/"), dict(request.query_params)

    def link(number, rel):
        return f'<{address}/network/readings?{urlencode({**query, "page": number})}>; rel="{rel}"'

    links = [link(page - 1, "prev")] if page > 1 else []
    links += [link(page + 1, "next"), link(last, "last")] if page < last else []
    links += [link(1, "first")] if page > 1 else []
    if links:
        response.headers["Link"] = ", ".join(links)
    start = (page - 1) * per_page
    return {"readings": chosen[start:start + per_page], "page": page, "per_page": per_page, "total": len(chosen)}


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/network/readings?station=oslo&per_page=30&page=2", timeout=10)
answer = response.json()
counts = {key: value for key, value in answer.items() if key != "readings"}
print(response.status_code, counts, "|", len(answer["readings"]), "readings")
for rel, link in response.links.items():
    print(f"  {rel}: {link['url']}")
for query in ["per_page=500", "station=narvik"]:
    response = requests.get(f"{app_url}/network/readings?{query}", timeout=10)
    answer = response.json()
    print(response.status_code, query, "|", answer.get("per_page", answer.get("error")))


200 {'page': 2, 'per_page': 30, 'total': 72} | 30 readings
  prev: http://127.0.0.1:8000/network/readings?station=oslo&per_page=30&page=1
  next: http://127.0.0.1:8000/network/readings?station=oslo&per_page=30&page=3
  last: http://127.0.0.1:8000/network/readings?station=oslo&per_page=30&page=3
  first: http://127.0.0.1:8000/network/readings?station=oslo&per_page=30&page=1
200 per_page=500 | 100
400 station=narvik | no station has the id 'narvik'


Page 2 of Oslo's 72 readings holds 30, and `Link` names the page before it, the next, the last and
the first, in the order the practice API sends them, with `station` and `per_page` kept in every
address. A `per_page` of 500 got 100, and a station that does not exist got `400`, as they do from
the practice API.

### The practice API, rebuilt

The pieces of this notebook, in one app: the stations, the readings with their pages, and the plans
with every method, with every error in the practice API's shape. Its `plans`, `plan_ids` and
`saved_posts` start empty, as the practice API's plans do:


In [10]:
app = FastAPI(title="The practice API, rebuilt")
plans = {}
plan_ids = itertools.count(1)
saved_posts = {}


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    """Answer every HTTPException in the practice API's shape, keeping its headers."""
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.exception_handler(RequestValidationError)
def plan_problems(request, error):
    """Answer a request that broke the rules with its problems, as the practice API does."""
    problems = [{"field": ".".join(str(part) for part in problem["loc"][1:]) or None, "problem": problem["msg"]}
                for problem in error.errors()]
    return JSONResponse(status_code=422, content={"error": "the plan has problems", "problems": problems})


@app.get("/stations")
def list_stations():
    return [{"id": found["id"], "name": found["name"]} for found in practice_api.STATIONS.values()]


@app.get("/stations/{station_id}")
def read_station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


@app.get("/network/readings")
def readings(request: Request, response: Response, station: str | None = None,
             page: Annotated[int, Query(ge=1)] = 1, per_page: Annotated[int, Query(ge=1)] = 30):
    if station is not None and station not in practice_api.STATIONS:
        raise HTTPException(status_code=400, detail=f"no station has the id {station!r}")
    per_page = min(per_page, 100)
    chosen = [reading for reading in practice_api.READINGS if station in (None, reading["station"])]
    last = max(1, math.ceil(len(chosen) / per_page))
    address, query = str(request.base_url).rstrip("/"), dict(request.query_params)

    def link(number, rel):
        return f'<{address}/network/readings?{urlencode({**query, "page": number})}>; rel="{rel}"'

    links = [link(page - 1, "prev")] if page > 1 else []
    links += [link(page + 1, "next"), link(last, "last")] if page < last else []
    links += [link(1, "first")] if page > 1 else []
    if links:
        response.headers["Link"] = ", ".join(links)
    start = (page - 1) * per_page
    return {"readings": chosen[start:start + per_page], "page": page, "per_page": per_page, "total": len(chosen)}


@app.get("/network/plans")
def list_plans():
    return [plans[plan_id] for plan_id in sorted(plans)]


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response, idempotency_key: Annotated[str | None, Header()] = None):
    fields = plan.model_dump(mode="json")
    if idempotency_key in saved_posts:
        sent, created = saved_posts[idempotency_key]
        if sent != fields:
            raise HTTPException(status_code=422, detail="this Idempotency-Key was already used with a different body")
        response.headers["Idempotent-Replayed"] = "true"
    else:
        created = plan_from(next(plan_ids), fields)
        plans[created["id"]] = created
        if idempotency_key is not None:
            saved_posts[idempotency_key] = (fields, created)
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


@app.get("/network/plans/{plan_id}")
def read_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    return plan


@app.put("/network/plans/{plan_id}")
def replace_plan(new: Plan, plan: Annotated[dict, Depends(plan_or_404)]):
    plans[plan["id"]] = plan_from(plan["id"], new.model_dump(mode="json"))
    return plans[plan["id"]]


@app.patch("/network/plans/{plan_id}")
def change_plan(change: PlanChange, plan: Annotated[dict, Depends(plan_or_404)]):
    plans[plan["id"]] = plan_from(plan["id"], {**plan, **change.model_dump(mode="json", exclude_unset=True)})
    return plans[plan["id"]]


@app.delete("/network/plans/{plan_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    del plans[plan["id"]]


app_url = practice_api.serve(app)
print(app.title, "|", len([route for route in app.routes if route.path.startswith(("/stations", "/network"))]), "routes")


The practice API, rebuilt | 9 routes


The same requests now go to the practice API and to the app. Each line says whether the two answers
had the same status code, the same body, and the same `Location`, `Link`, `Allow` and
`Idempotent-Replayed` headers, with each server's own address taken out of `Link`. A `422`'s problems
are compared by field, since the messages are each server's own:


In [11]:
def outcome(response, address):
    """What a response tells a client: its status code, its body, and the headers a client follows."""
    answer = response.json() if response.content else None
    if response.status_code == 422 and "problems" in answer:
        answer = [problem["field"] for problem in answer["problems"]]
    names = ("Location", "Link", "Allow", "Idempotent-Replayed")
    headers = {name: response.headers[name].replace(address, "") for name in names if name in response.headers}
    return response.status_code, answer, headers


key = {"Idempotency-Key": str(uuid.uuid4())}
alta = {"name": "Alta", "latitude": 69.97, "longitude": 23.27, "elevation_m": 8}
steps = [("GET", "/stations/tromso", None, None),
         ("GET", "/stations/narvik", None, None),
         ("GET", "/network/readings?station=oslo&per_page=30&page=3", None, None),
         ("GET", "/network/readings?station=narvik", None, None),
         ("POST", "/network/plans", alta, key),
         ("POST", "/network/plans", alta, key),
         ("POST", "/network/plans", {"name": "Lakselv", "latitude": 170.05}, None),
         ("PATCH", "/network/plans/1", {"elevation_m": None, "opens": "2027-06-01"}, None),
         ("PUT", "/network/plans/1", {"name": "Alta", "latitude": 69.96, "longitude": 23.24}, None),
         ("GET", "/network/plans", None, None),
         ("DELETE", "/network/plans/1", None, None),
         ("DELETE", "/network/plans/1", None, None),
         ("POST", "/stations/tromso", None, None),
         ("POST", "/network/plans/1", None, None)]

for method, path, body, headers in steps:
    theirs = requests.request(method, f"{BASE}{path}", json=body, headers=headers, timeout=10)
    ours = requests.request(method, f"{app_url}{path}", json=body, headers=headers, timeout=10)
    same = outcome(theirs, BASE) == outcome(ours, app_url)
    print(f"{method:<6} {path:<50} {theirs.status_code} {ours.status_code} {'same' if same else 'different'}")
    if not same:
        print("    practice API:", outcome(theirs, BASE))
        print("    this app:    ", outcome(ours, app_url))


GET    /stations/tromso                                   200 200 same
GET    /stations/narvik                                   404 404 same
GET    /network/readings?station=oslo&per_page=30&page=3  200 200 same
GET    /network/readings?station=narvik                   400 400 same
POST   /network/plans                                     201 201 same
POST   /network/plans                                     201 201 same
POST   /network/plans                                     422 422 same
PATCH  /network/plans/1                                   200 200 same
PUT    /network/plans/1                                   200 200 same
GET    /network/plans                                     200 200 same
DELETE /network/plans/1                                   204 204 same
DELETE /network/plans/1                                   404 404 same
POST   /stations/tromso                                   405 405 different
    practice API: (405, {'error': 'POST not allowed: the stations are re

### Where each part came from

| In the app | What it relies on | The section that showed it |
|---|---|---|
| `status_code=status.HTTP_201_CREATED` and `HTTP_204_NO_CONTENT` | a status for each outcome, named | A status code for every outcome, and a Location header |
| `response.headers["Location"]` | headers set through a `Response` parameter | A status code for every outcome, and a Location header |
| `Depends(plan_or_404)` in four routes | a lookup and a `404` written once | One plan and every plan: GET, and a dependency |
| `new: Plan` in `replace_plan` | a `PUT` body that is the whole plan | Replacing a plan: PUT |
| `exclude_unset=True` in `change_plan` | only the fields a `PATCH` sent | Changing part of a plan: PATCH |
| `idempotency_key` with `Header()` | a request header read by name | A header the route reads: Idempotency-Key |
| `@app.exception_handler(StarletteHTTPException)` | every error in one shape | Every error in one shape |
| `@app.exception_handler(RequestValidationError)` | problems in the practice API's shape | the **Validating Requests** notebook |
| `request.base_url` and `request.query_params` | a `Link` header built from the request | Pages with a Link header |

Twelve of the fourteen requests got the same answer from both servers, from a created plan's
`Location` to a page's `Link`, a replayed `POST` and a second `DELETE`. The two that differ are both
`405`s. The practice API names the method in its message, and lists every method a plan takes in
`Allow`, while Starlette answers `Method Not Allowed`, and lists only the methods of the first route
whose path matched. A client that reads the status code cannot tell the servers apart. A client that
reads `Allow` to find a method that works would be misled by the rebuild, and that is the kind of
difference that comparing two servers, request by request, exists to find.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/17-a-complete-api-solutions.ipynb).

**1.** Make a dependency, `station_or_404`, that takes `station_id` from the path and returns the
station from `practice_api.STATIONS`, or raises a `404` with the detail `no station with id ...`. Use
it in a route, `GET /stations/{station_id}`, and request `tromso` and `bodo`.


In [12]:
# your code here


**2.** Make a dictionary of notes, and a route, `POST /stations/{station_id}/notes`, whose body is a
model `Note` with a `text`. It answers `201` with the note, as `{"id": ..., "station": ..., "text":
...}`, and the note's address, `/stations/{station_id}/notes/{id}`, in `Location`, and it uses
`station_or_404`. Post a note for `oslo`, and print the status code, `Location` and the body.


In [13]:
# your code here


**3.** Make a dependency, `note_or_404`, and a route, `GET /stations/{station_id}/notes/{note_id}`,
that uses both dependencies. Request the note from task 2, and note 9.


In [14]:
# your code here


**4.** Make a route, `DELETE /stations/{station_id}/notes/{note_id}`, that answers `204`. Send it
twice for the note from task 2, and print both status codes.


In [15]:
# your code here


**5.** Make an app with a handler for Starlette's `HTTPException` that answers `{"error": ...}`, and
the route from task 3. Request note 9, and the path `/stations/oslo/note/1`, and print both answers.


In [16]:
# your code here


**6.** Make an app with the handler from task 5 and the route from task 1. Send
`GET /stations/tromso` and `GET /stations/bodo` to it and to the practice API, and print whether
each pair of answers has the same status code and the same body.


In [17]:
# your code here


## Common errors

### TypeError: plan_or_404() missing 1 required positional argument: 'plan_id'


In [18]:
app = FastAPI()


@app.get("/network/plans/{plan_id}")
def read_plan(plan: Annotated[dict, Depends(plan_or_404())]):
    return plan


TypeError: plan_or_404() missing 1 required positional argument: 'plan_id'

`plan_or_404()` called the function on the spot, while Python was defining `read_plan`, with no plan
id to give it. `Depends` takes the function itself, and FastAPI calls it for every request, with the
parameters it needs from that request. FastAPI's documentation puts it plainly: pass the dependency,
and do not add the parentheses at the end. Name the function without calling it:


In [19]:
app = FastAPI()


@app.get("/network/plans/{plan_id}")
def read_plan(plan: Annotated[dict, Depends(plan_or_404)]):
    return plan


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/network/plans/1", timeout=10)
print(response.status_code, response.json())


404 {'detail': 'no plan has the id 1'}


The dependency ran, and found no plan 1, since the rebuilt API removed it, so it answered `404`.

### No error, and the rest of the plan gone: a PATCH without exclude_unset


In [20]:
plans[5] = {"id": 5, "name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation_m": 20}
app = FastAPI()


@app.patch("/network/plans/{plan_id}")
def change_plan(change: PlanChange, plan: Annotated[dict, Depends(plan_or_404)]):
    sent = change.model_dump(mode="json")                      # every field, sent or not
    plans[plan["id"]] = plan_from(plan["id"], {**plan, **sent})
    return plans[plan["id"]]


app_url = practice_api.serve(app)
response = requests.patch(f"{app_url}/network/plans/5", json={"elevation_m": 25}, timeout=10)
print(response.status_code, response.json())


200 {'id': 5, 'elevation_m': 25.0}


The request changed the elevation, and the plan lost its name and its coordinates. `model_dump()`
holds every field of `PlanChange`, and a field the request did not send holds its default, `None`, so
the merge replaced the name, the latitude and the longitude with `None`, and `plan_from` left them
out. Keep only the fields the request set, with `exclude_unset=True`:


In [21]:
plans[5] = {"id": 5, "name": "Hasvik", "latitude": 70.49, "longitude": 22.14, "elevation_m": 20}
app = FastAPI()


@app.patch("/network/plans/{plan_id}")
def change_plan(change: PlanChange, plan: Annotated[dict, Depends(plan_or_404)]):
    sent = change.model_dump(mode="json", exclude_unset=True)
    plans[plan["id"]] = plan_from(plan["id"], {**plan, **sent})
    return plans[plan["id"]]


app_url = practice_api.serve(app)
response = requests.patch(f"{app_url}/network/plans/5", json={"elevation_m": 25}, timeout=10)
print(response.status_code, response.json())


200 {'id': 5, 'name': 'Hasvik', 'latitude': 70.49, 'longitude': 22.14, 'elevation_m': 25.0}


### No error, and {'detail': 'Not Found'}: a handler for FastAPI's HTTPException, not Starlette's


In [22]:
app = FastAPI()


@app.exception_handler(HTTPException)
def error_body(request, error):
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/stations/{station_id}")
def read_station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
for path in ["/stations/narvik", "/station/tromso"]:
    print(path, requests.get(f"{app_url}{path}", timeout=10).json())


/stations/narvik {'error': "no station with id 'narvik'"}
/station/tromso {'detail': 'Not Found'}


The route's `404` came back in the practice API's shape, and the `404` for a mistyped path did not.
The route raised FastAPI's `HTTPException`, which the handler catches. A path that matches no route
is answered by Starlette, which raises its own `HTTPException`, and a handler for FastAPI's subclass
never sees it. Register the handler for Starlette's class:


In [23]:
app = FastAPI()


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/stations/{station_id}")
def read_station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
for path in ["/stations/narvik", "/station/tromso"]:
    print(path, requests.get(f"{app_url}{path}", timeout=10).json())


/stations/narvik {'error': "no station with id 'narvik'"}
/station/tromso {'error': 'Not Found'}


### 422, Field required: a Response parameter without its type


In [24]:
app = FastAPI()


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response):
    created = plan_from(next(plan_ids), plan.model_dump(mode="json"))
    plans[created["id"]] = created
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
body = {"name": "Alta", "latitude": 69.97, "longitude": 23.27}
response = requests.post(f"{app_url}/network/plans", json=body, timeout=10)
print(response.status_code, response.json())


422 {'detail': [{'type': 'missing', 'loc': ['query', 'response'], 'msg': 'Field required', 'input': None}]}


FastAPI decides what a parameter is from its type. `response` has none, so FastAPI took it for a
query parameter, a required one, and the request did not send it: `loc` says `["query", "response"]`.
The route never ran. Type the parameter as `Response`, and FastAPI passes the route its temporary
response instead:


In [25]:
app = FastAPI()


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response):
    created = plan_from(next(plan_ids), plan.model_dump(mode="json"))
    plans[created["id"]] = created
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/network/plans", json=body, timeout=10)
print(response.status_code, response.headers["Location"])


201 /network/plans/2


### No error, and 200 for a created plan: a route with no status_code


In [26]:
app = FastAPI()


@app.post("/network/plans")
def create_plan(plan: Plan, response: Response):
    created = plan_from(next(plan_ids), plan.model_dump(mode="json"))
    plans[created["id"]] = created
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/network/plans", json=body, timeout=10)
print(response.status_code, response.headers["Location"])


200 /network/plans/3


The plan was created, and the answer said `200 OK`, FastAPI's default, where the practice API says
`201 Created`. A client that checks for `201`, as a client written for the practice API may, takes
the new plan for a failure, the opposite of the mistake the **Sending Data** notebook warned of. Give
the route its status:


In [27]:
app = FastAPI()


@app.post("/network/plans", status_code=status.HTTP_201_CREATED)
def create_plan(plan: Plan, response: Response):
    created = plan_from(next(plan_ids), plan.model_dump(mode="json"))
    plans[created["id"]] = created
    response.headers["Location"] = f"/network/plans/{created['id']}"
    return created


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/network/plans", json=body, timeout=10)
print(response.status_code, response.headers["Location"])


201 /network/plans/4


## Recap

- `status_code=` in a route's decorator, or a name from `fastapi.status`, sets its status for
  success: `201` for a created item, and `204`, with no body, for a removed one. `200` is the
  default.
- A parameter typed `Response` lets a route set headers such as `Location` and `Link`, which FastAPI
  copies into the response it sends, and one typed `Request` gives it the request.
- A dependency named in `Depends` runs before the route, with parameters of its own from the request,
  and can end the request with an `HTTPException`, so one `plan_or_404` serves four routes.
- `PUT` replaces all of an item. `PATCH` merges only `model_dump(exclude_unset=True)`, so a field the
  request did not send keeps its value.
- `Header()` reads a request header by the parameter's name, with underscores as hyphens.
- A handler for Starlette's `HTTPException` answers every error, a route's or FastAPI's own, in one
  shape.
- Comparing a rebuilt API with its original, request by request, finds every place they differ.


## What is next

The **Hosting an API** notebook. The apps in this notebook answer only on Colab's machine, and only
while the notebook runs. That notebook turns an app into files that a host can run, gives it a start
command and a secret, and puts it on a public address with a free host.


---

&#8592; **Previous:** [Validating Requests](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/16-validating-requests.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Hosting an API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/18-hosting-an-api.ipynb) &#8594;
